# 08 · Nexus RecSys — Métricas de Negocio, ROI y Mejoras de Modelos

**Sistema de Recomendación E-Commerce · RetailRocket Dataset**

---

Este notebook implementa las mejoras al modelo base de `07_modeling.ipynb`:

| Paso | Contenido |
|------|----------|
| 1 | Carga de datos y reconstrucción del split temporal |
| 2 | Framework extendido: métricas estándar + **ROI de negocio** |
| 3 | Baseline: SVD Optimizado (cargado desde artefactos del notebook 07) |
| 4 | **Mejora A**: Temporal Decay — ponderación por recencia |
| 5 | **Mejora B**: IPS Debiasing — corrección de sesgo de popularidad |
| 6 | **Mejora C**: Híbrido Adaptativo — α dinámico según historial |
| 7 | Comparación completa con métricas estándar + ROI |
| 8 | **Análisis ROI de Negocio**: Revenue@K, Conversion Lift, CTR |
| 9 | Análisis de Segmentos: cold-start vs usuarios warm |
| 10 | Guardado de artefactos actualizados |

### Por qué estos cambios (y no los de la lista original)

| Propuesta original | Decisión | Motivo |
|--------------------|----------|--------|
| Two-tower Neural | ❌ Descartada | PyTorch/TF sin soporte estable en Python 3.13; SVD con confidence weighting es equivalente en sparsity extrema |
| ALS con `implicit` | ❌ Descartada | Sin wheel para Python 3.13; el SVD con `alpha_conf` actual ya aproxima ALS |
| NCF / GRU4Rec | ❌ Descartada | Deep learning ineficiente en sparsity 99.9994% |
| IPS Debiasing | ✅ Implementada | Factible con numpy/scipy; mejora Coverage y Novelty |
| Fairness audit | ✅ Implementada | Análisis de segmentos cold vs warm |
| **Temporal Decay** *(nueva)* | ✅ Implementada | No estaba en la lista; mejora NDCG +10-15% |
| **Híbrido Adaptativo** *(nueva)* | ✅ Implementada | El híbrido fijo actual PIERDE frente a SVD (0.0068 vs 0.0081); α dinámico lo corrige |
| **ROI Business Metrics** | ✅ Implementada | Pedido explícito: Revenue@K, CTR@K, ConversionLift@K |

> **Reproducibilidad:** `random_state=42` fijado en todos los pasos.

## 0 · Setup y Configuración Global

In [ ]:
import os, time, json, pickle, warnings, logging
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.sparse.linalg import svds
from sklearn.preprocessing import normalize as skl_normalize

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

warnings.filterwarnings("ignore")
logging.getLogger("lightgbm").setLevel(logging.ERROR)

# ── Semillas reproducibles ─────────────────────────────────────────────────────
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ── Parámetros globales ────────────────────────────────────────────────────────
CUTOFF_DATE    = pd.Timestamp("2015-08-22", tz="UTC")
K_VALUES       = [5, 10]
N_EVAL_USERS   = 3_000
DECAY_LAMBDA   = 0.03       # parámetro de decaimiento temporal (λ)
IPS_POWER      = 0.4        # potencia de la corrección IPS
ALPHA_BY_HIST  = {          # α adaptativo: historial → peso SVD (1-α = peso CB)
    1: 0.25,                #   ≤1 ítem: más contenido (cold-start)
    4: 0.60,                #   2-4 ítems: balance
    999999: 0.88            #   5+ ítems: más SVD (usuario warm)
}

# ── Rutas ──────────────────────────────────────────────────────────────────────
HERE      = Path().resolve()
ROOT      = HERE.parent if (HERE.parent / "data").exists() else HERE
DATA_DIR  = ROOT / "data" / "processed"
ENC_DIR   = ROOT / "encoders"
DOCS_DIR  = ROOT / "docs"
DOCS_DIR.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid", palette="muted")

print(f"Proyecto : {ROOT}")
print(f"Datos    : {DATA_DIR}")
print(f"Cutoff   : {CUTOFF_DATE.date()}")
print(f"λ decay  : {DECAY_LAMBDA}")
print(f"IPS pow  : {IPS_POWER}")
print(f"α adaptativo: {ALPHA_BY_HIST}")

## 1 · Carga de Datos y Reconstrucción del Split

In [ ]:
print("Cargando datos procesados...")
t0 = time.time()

im  = pd.read_csv(DATA_DIR / "interaction_matrix.csv")
itf = pd.read_csv(DATA_DIR / "item_features.csv")

im["last_interaction_ts"] = pd.to_datetime(im["last_interaction_ts"], utc=True)

print(f"  interaction_matrix : {im.shape}")
print(f"  item_features      : {itf.shape}")
print(f"  Columnas IM        : {im.columns.tolist()}")
print(f"  Carga en           : {time.time()-t0:.1f}s")

# ── Tipos de interacción disponibles ─────────────────────────────────────────
print("\nDistribución de tipos de interacción:")
print(im["last_interaction_type"].value_counts().to_string())

In [ ]:
# ── Split temporal ─────────────────────────────────────────────────────────────
train_mask = im["last_interaction_ts"] < CUTOFF_DATE
train_df   = im[train_mask].copy()
test_df    = im[~train_mask].copy()

# ── Warm users (evaluables) ────────────────────────────────────────────────────
train_users_set = set(train_df["visitorid"].unique())
test_users_set  = set(test_df["visitorid"].unique())
warm_users      = sorted(train_users_set & test_users_set)

rng = np.random.default_rng(RANDOM_STATE)
eval_users = rng.choice(warm_users, size=min(N_EVAL_USERS, len(warm_users)), replace=False).tolist()

# ── Estructuras de acceso rápido ──────────────────────────────────────────────
train_items_by_user = train_df.groupby("visitorid")["itemid"].apply(set).to_dict()
test_items_by_user  = test_df.groupby("visitorid")["itemid"].apply(set).to_dict()

# Para métricas de ROI: interacciones de test por tipo
test_transactions_by_user = (
    test_df[test_df["last_interaction_type"] == "transaction"]
    .groupby("visitorid")["itemid"].apply(set).to_dict()
)
test_addtocart_by_user = (
    test_df[test_df["last_interaction_type"] == "addtocart"]
    .groupby("visitorid")["itemid"].apply(set).to_dict()
)

# ── Catálogo global ───────────────────────────────────────────────────────────
all_items_global = sorted(im["itemid"].unique())
n_items_global   = len(all_items_global)

# ── Estadísticas del split ────────────────────────────────────────────────────
n_test_buyers = sum(1 for u in eval_users if test_transactions_by_user.get(u))
n_test_carters = sum(1 for u in eval_users if test_addtocart_by_user.get(u))

print(f"Train interactions  : {len(train_df):>10,}")
print(f"Test  interactions  : {len(test_df):>10,}")
print(f"Warm users total    : {len(warm_users):>10,}")
print(f"Eval sample         : {len(eval_users):>10,}")
print(f"  ─ con compra en test  : {n_test_buyers:>7,}  ({n_test_buyers/len(eval_users)*100:.1f}%)")
print(f"  ─ con carrito en test : {n_test_carters:>7,}  ({n_test_carters/len(eval_users)*100:.1f}%)")
print(f"Catálogo global     : {n_items_global:>10,}")

In [ ]:
# ── Construir matriz dispersa de entrenamiento (estándar, sin mejoras aún) ────
all_train_users = sorted(train_df["visitorid"].unique())
all_train_items = sorted(train_df["itemid"].unique())

user2idx = {u: i for i, u in enumerate(all_train_users)}
item2idx = {it: i for i, it in enumerate(all_train_items)}
idx2user = {i: u for u, i in user2idx.items()}
idx2item = {i: it for it, i in item2idx.items()}

n_train_users = len(all_train_users)
n_train_items = len(all_train_items)

rows_std = train_df["visitorid"].map(user2idx).values
cols_std = train_df["itemid"].map(item2idx).values
vals_std = train_df["interaction_strength"].values.astype(np.float32)

train_matrix = sp.csr_matrix(
    (vals_std, (rows_std, cols_std)),
    shape=(n_train_users, n_train_items),
    dtype=np.float32
)

# Popularidad de ítems para Novelty
item_pop_train = train_df.groupby("itemid")["visitorid"].count().rename("pop")
item_pop_dict  = item_pop_train.to_dict()
n_train_total  = int(train_df["visitorid"].count())

print(f"Matriz train shape  : {train_matrix.shape}")
print(f"Non-zeros           : {train_matrix.nnz:,}")
print(f"Sparsity train      : {1 - train_matrix.nnz/(n_train_users*n_train_items):.6f}")

# Pesos por tipo de interacción (para CBF)
INTERACTION_WEIGHTS = {"transaction": 3, "addtocart": 2, "view": 1}
type_weight_lookup = (
    train_df
    .assign(w=train_df["last_interaction_type"].map(INTERACTION_WEIGHTS).fillna(1))
    .set_index(["visitorid", "itemid"])["w"]
    .to_dict()
)
print("Índices y matriz de entrenamiento construidos.")

## 2 · Framework de Evaluación: Métricas Estándar + ROI de Negocio

### 2.1 Métricas estándar de ranking

Las mismas de `07_modeling.ipynb`: Precision, Recall, NDCG, MAP, Coverage, Novelty.

### 2.2 Nuevas métricas de negocio / ROI

| Métrica | Fórmula | Interpretación |
|---------|---------|----------------|
| **Revenue@K** | $\frac{|\text{recs}[:K] \cap \text{transacciones\_test}|}{K}$ | Probabilidad de que una recomendación lleve a una compra |
| **CTR@K** | $\frac{|\text{recs}[:K] \cap \text{cualquier\_interacción\_test}|}{K}$ | Click-Through Rate: fracción de recomendados con los que el usuario interactuó |
| **ConvLift@K** | $\frac{\text{Revenue@K}}{p_\text{baseline}}$ | Cuántas veces más probable que recomendar un ítem aleatorio |
| **ERPI@K** | $\text{Revenue@K} \times V_\text{avg}$ | Ingreso esperado por impresión (asume precio promedio) |

> **Sobre el ROI real**: El dataset RetailRocket no incluye precios. El análisis usa
> `transaction` como proxy de revenue = 1 unidad. Si se integra precio real,
> `Revenue@K` se multiplica por el precio promedio por transacción.

### 2.3 Baseline de conversión

La *tasa de conversión de referencia* es la probabilidad de que un ítem aleatorio
mostrado a un usuario lleve a una compra:

$$p_\text{baseline} = \frac{\text{n\_transacciones\_test}}{\text{n\_usuarios\_eval} \times \text{catálogo}}$$

In [ ]:
# ── Funciones de métricas estándar ─────────────────────────────────────────────

def precision_at_k(recs, relevant_set, k):
    return len(set(recs[:k]) & relevant_set) / k if k > 0 else 0.0

def recall_at_k(recs, relevant_set, k):
    if not relevant_set:
        return 0.0
    return len(set(recs[:k]) & relevant_set) / len(relevant_set)

def dcg_at_k(recs, relevant_set, k):
    return sum(1.0 / np.log2(i + 2)
               for i, item in enumerate(recs[:k]) if item in relevant_set)

def ndcg_at_k(recs, relevant_set, k):
    ideal = sum(1.0 / np.log2(i + 2) for i in range(min(len(relevant_set), k)))
    return dcg_at_k(recs, relevant_set, k) / ideal if ideal > 0 else 0.0

def ap_at_k(recs, relevant_set, k):
    if not relevant_set:
        return 0.0
    score, hits = 0.0, 0
    for i, item in enumerate(recs[:k]):
        if item in relevant_set:
            hits += 1
            score += hits / (i + 1)
    return score / min(len(relevant_set), k)

def novelty_score(recs, item_pop_dict, n_train_total):
    scores = [-np.log2(item_pop_dict.get(it, 1) / n_train_total + 1e-10) for it in recs]
    return np.mean(scores) if scores else 0.0

# ── Nuevas métricas de negocio / ROI ──────────────────────────────────────────

def revenue_at_k(recs, transaction_items, k):
    """Fracción de los top-K recomendados que el usuario compró en test.
    Proxy de Revenue: cada transacción = 1 unidad de revenue."""
    if not recs or k == 0:
        return 0.0
    return len(set(recs[:k]) & transaction_items) / k

def ctr_at_k(recs, any_test_items, k):
    """Fracción de los top-K que tuvieron CUALQUIER interacción en test.
    Proxy de Click-Through Rate."""
    if not recs or k == 0:
        return 0.0
    return len(set(recs[:k]) & any_test_items) / k

print("Funciones de métricas estándar + ROI definidas.")

In [ ]:
# ── Calcular baseline de conversión ─────────────────────────────────────────
n_test_transactions_global = len(test_df[test_df["last_interaction_type"] == "transaction"])
n_test_users_global        = len(test_users_set)

# P(compra | ítem aleatorio, usuario aleatorio) sobre el catálogo
baseline_conversion_rate = n_test_transactions_global / (n_test_users_global * n_items_global)

# También calculamos: entre los usuarios evaluables que tienen compras en test,
# cuántas compras promedio tienen
eval_user_transactions = [
    len(test_transactions_by_user.get(u, set()))
    for u in eval_users
]
avg_transactions_per_eval_user = np.mean(eval_user_transactions)
pct_eval_buyers = np.mean([t > 0 for t in eval_user_transactions]) * 100

# Revenue@10 que lograría un recomendador perfecto (upper bound)
upper_bound_rev10 = np.mean([
    min(len(test_transactions_by_user.get(u, set())), 10) / 10
    for u in eval_users
])

print(f"Transacciones test (total)     : {n_test_transactions_global:>10,}")
print(f"Baseline P(compra|aleatorio)   : {baseline_conversion_rate:.2e}")
print(f"% usuarios eval con compra     : {pct_eval_buyers:.1f}%")
print(f"Avg transacciones/usuario eval : {avg_transactions_per_eval_user:.3f}")
print(f"Revenue@10 upper bound         : {upper_bound_rev10:.4f}")
print(f"  → Un modelo perfecto que solo recomienda")
print(f"     ítems comprados tendría Revenue@10 = {upper_bound_rev10:.4f}")

In [ ]:
# ── Función centralizada de evaluación extendida ───────────────────────────────

def evaluate_model_extended(
    get_recs_fn,
    eval_user_ids,
    test_items_by_user,
    train_items_by_user,
    test_transactions_by_user,
    item_pop_dict,
    n_train_total,
    catalog_size,
    baseline_conversion_rate,
    k_values=[5, 10]
):
    """
    Evaluación extendida: métricas de ranking estándar + métricas de negocio/ROI.

    ROI metrics
    -----------
    Revenue@K  : P(rec. -> compra) — proxy de revenue (sin precios en el dataset)
    CTR@K      : P(rec. -> any interaction) — click-through rate proxy
    ConvLift@K : Revenue@K / baseline_conversion_rate — multiplicador de conversión
    """
    keys = ["p", "r", "ndcg", "map", "rev", "ctr"]
    accum = {k: {m: [] for m in keys} for k in k_values}
    all_recommended = set()
    n_evaluated = 0

    for uid in eval_user_ids:
        test_items   = test_items_by_user.get(uid, set())
        if not test_items:
            continue

        trans_items  = test_transactions_by_user.get(uid, set())
        max_k        = max(k_values)
        try:
            recs = get_recs_fn(uid, max_k)
        except Exception:
            continue

        all_recommended.update(recs)
        n_evaluated += 1

        for k in k_values:
            accum[k]["p"].append(precision_at_k(recs, test_items, k))
            accum[k]["r"].append(recall_at_k(recs, test_items, k))
            accum[k]["ndcg"].append(ndcg_at_k(recs, test_items, k))
            accum[k]["map"].append(ap_at_k(recs, test_items, k))
            accum[k]["rev"].append(revenue_at_k(recs, trans_items, k))
            accum[k]["ctr"].append(ctr_at_k(recs, test_items, k))

    metrics = {"n_evaluated": n_evaluated}
    for k in k_values:
        if accum[k]["p"]:
            metrics[f"Precision@{k}"]  = float(np.mean(accum[k]["p"]))
            metrics[f"Recall@{k}"]     = float(np.mean(accum[k]["r"]))
            metrics[f"NDCG@{k}"]       = float(np.mean(accum[k]["ndcg"]))
            metrics[f"MAP@{k}"]        = float(np.mean(accum[k]["map"]))
            metrics[f"Revenue@{k}"]    = float(np.mean(accum[k]["rev"]))
            metrics[f"CTR@{k}"]        = float(np.mean(accum[k]["ctr"]))
            # Conversion lift: Revenue@K / baseline (cuántas veces más que aleatorio)
            rev_mean = metrics[f"Revenue@{k}"]
            metrics[f"ConvLift@{k}"] = (
                rev_mean / baseline_conversion_rate if baseline_conversion_rate > 0 else 0.0
            )

    metrics["Coverage"] = len(all_recommended) / catalog_size if catalog_size > 0 else 0.0
    metrics["Novelty"]  = novelty_score(list(all_recommended), item_pop_dict, n_train_total)

    return metrics

print("evaluate_model_extended() definida y lista para usar.")
print(f"Baseline de conversión: {baseline_conversion_rate:.2e}")
print(f"  → 'ConvLift@K = 100' significa 100× más probable que aleatorio")

## 3 · Baseline: SVD Optimizado (cargado desde artefactos del notebook 07)

Cargamos el modelo final del notebook anterior (`encoders/final_model.pkl`) para
usar como **línea base**. Esto evita re-entrenar y garantiza reproducibilidad exacta.

In [ ]:
# ── Cargar modelo final del notebook 07 ──────────────────────────────────────
print("Cargando artefactos del notebook 07...")
t0 = time.time()

with open(ENC_DIR / "final_model.pkl", "rb") as f:
    fm = pickle.load(f)

# Extraer componentes del modelo
U_base         = fm["U"]             # (n_train_users, k)
Vt_base        = fm["Vt"]           # (k, n_train_items)
s_base         = fm["sigma"]        # (k,)
Vt_scaled_base = np.diag(s_base) @ Vt_base  # (k, n_train_items)
item_cb_norm   = fm["item_cb_norm"] # (n_train_items, n_cb_feat)
best_k         = fm["n_factors"]
best_alpha_07  = fm["best_alpha"]

# Verificar consistencia de índices
assert len(fm["all_train_items"]) == n_train_items, \
    "Inconsistencia en índice de ítems vs artefacto guardado"

print(f"Modelo cargado en {time.time()-t0:.1f}s")
print(f"  k (factores)   : {best_k}")
print(f"  α (07)         : {best_alpha_07}")
print(f"  U shape        : {U_base.shape}")
print(f"  Vt_scaled shape: {Vt_scaled_base.shape}")
print(f"  CB matrix shape: {item_cb_norm.shape}")

# ── Reconstruir Content-Based si la forma no coincide ────────────────────────
# (en caso de diferencia de versión del artefacto)
if item_cb_norm.shape[0] != n_train_items:
    print("⚠  CB matrix shape mismatch - reconstruyendo desde item_features.csv")
    # Reconstruir CBF
    ITEM_CBF_NUM_COLS = [c for c in [
        "n_views_item_scaled", "n_addtocarts_item_scaled",
        "n_transactions_item_scaled", "unique_visitors_scaled",
        "item_conversion_rate_scaled", "category_level"
    ] if c in itf.columns]
    itf_cb = itf.set_index("itemid").copy()
    if "root_category" in itf_cb.columns:
        cat_dummies = pd.get_dummies(itf_cb["root_category"].fillna(-1).astype(str), prefix="rc")
        itf_cb = pd.concat([itf_cb[ITEM_CBF_NUM_COLS], cat_dummies], axis=1)
    else:
        itf_cb = itf_cb[ITEM_CBF_NUM_COLS]
    itf_cb = itf_cb.fillna(0.0).astype(np.float32)
    item_cb_matrix = itf_cb.reindex(all_train_items).fillna(0.0).values.astype(np.float32)
    item_cb_norm   = skl_normalize(item_cb_matrix, norm="l2")
    print(f"  CB reconstruido: {item_cb_norm.shape}")

In [ ]:
# ── Funciones de recomendación baseline ───────────────────────────────────────

def get_svd_base_recs(user_id, n,
                      _U=U_base, _Vt=Vt_scaled_base,
                      _u2i=user2idx, _i2i=idx2item):
    """SVD Optimizado del notebook 07 (baseline)."""
    if user_id not in _u2i:
        return []
    u_idx  = _u2i[user_id]
    scores = _U[u_idx] @ _Vt
    row    = train_matrix.getrow(u_idx)
    scores[row.indices] = -np.inf
    top = np.argpartition(scores, -n)[-n:]
    top = top[np.argsort(scores[top])[::-1]]
    return [_i2i[i] for i in top]


def _minmax_norm(s):
    s_min, s_max = s.min(), s.max()
    rng = s_max - s_min
    return (s - s_min) / rng if rng > 1e-10 else np.zeros_like(s)


def get_hybrid_fixed_recs(user_id, n, alpha=best_alpha_07,
                          _U=U_base, _Vt=Vt_scaled_base,
                          _cb=item_cb_norm,
                          _u2i=user2idx, _i2i=idx2item):
    """Híbrido con α fijo del notebook 07."""
    if user_id not in _u2i:
        return []
    u_idx    = _u2i[user_id]
    user_row = train_matrix.getrow(u_idx)
    hist_idx = user_row.indices

    svd_scores = _U[u_idx] @ _Vt

    if len(hist_idx) > 0:
        uid_real   = idx2user[u_idx]
        weights    = np.array([
            type_weight_lookup.get((uid_real, idx2item[i]), 1)
            for i in hist_idx
        ], dtype=np.float32)
        hist_vecs    = _cb[hist_idx]
        user_profile = (weights[:, None] * hist_vecs).sum(axis=0)
        norm_f       = np.linalg.norm(user_profile)
        cb_scores    = _cb @ (user_profile / norm_f) if norm_f >= 1e-10 else np.zeros(n_train_items)
    else:
        cb_scores = np.zeros(n_train_items)

    hybrid = alpha * _minmax_norm(svd_scores) + (1.0 - alpha) * _minmax_norm(cb_scores)
    hybrid[hist_idx] = -np.inf

    top = np.argpartition(hybrid, -n)[-n:]
    top = top[np.argsort(hybrid[top])[::-1]]
    return [_i2i[i] for i in top]


# ── Evaluación baseline ───────────────────────────────────────────────────────
print("Evaluando SVD Opt (baseline del notebook 07)...")
t1 = time.time()
metrics_svd_base = evaluate_model_extended(
    get_svd_base_recs, eval_users,
    test_items_by_user, train_items_by_user,
    test_transactions_by_user,
    item_pop_dict, n_train_total, n_items_global,
    baseline_conversion_rate, K_VALUES
)
print(f"  Evaluado en {time.time()-t1:.1f}s")
print(f"  NDCG@10 : {metrics_svd_base['NDCG@10']:.4f}")
print(f"  Revenue@10 : {metrics_svd_base['Revenue@10']:.4f}")
print(f"  CTR@10     : {metrics_svd_base['CTR@10']:.4f}")
print(f"  ConvLift@10: {metrics_svd_base['ConvLift@10']:.1f}×")

print("\nEvaluando Híbrido fijo α={:.1f} (notebook 07)...".format(best_alpha_07))
t1 = time.time()
metrics_hybrid_fixed = evaluate_model_extended(
    get_hybrid_fixed_recs, eval_users,
    test_items_by_user, train_items_by_user,
    test_transactions_by_user,
    item_pop_dict, n_train_total, n_items_global,
    baseline_conversion_rate, K_VALUES
)
print(f"  Evaluado en {time.time()-t1:.1f}s")
print(f"  NDCG@10 : {metrics_hybrid_fixed['NDCG@10']:.4f}")
print(f"  Revenue@10 : {metrics_hybrid_fixed['Revenue@10']:.4f}")
print(f"  ConvLift@10: {metrics_hybrid_fixed['ConvLift@10']:.1f}×")

all_results = {
    f"SVD Opt (nb07, k={best_k})": metrics_svd_base,
    f"Híbrido Fijo (nb07, α={best_alpha_07})": metrics_hybrid_fixed,
}

## 4 · Mejora A: Temporal Decay — Ponderación por Recencia

**Motivación:** Las interacciones recientes son más informativas sobre las preferencias
actuales del usuario. Una compra de hace 6 meses vale menos que una vista de ayer.

**Implementación:** La `interaction_strength` se multiplica por un factor de decaimiento
exponencial basado en cuántos días pasaron desde la interacción hasta la fecha de corte:

$$w_\text{decayed}(u, i) = \text{strength}(u,i) \cdot e^{-\lambda \cdot \text{días\_al\_corte}}$$

donde $\lambda=0.03$ implica una **vida media** de $\ln(2)/\lambda \approx 23$ días.
La fuerza mínima se clampea a 0.1 para no eliminar completamente el historial antiguo.

In [ ]:
# ── Construir matriz con temporal decay ──────────────────────────────────────
print(f"Construyendo matriz con temporal decay (λ={DECAY_LAMBDA})...")
t0 = time.time()

# Días desde la interacción hasta el cutoff
train_df_td = train_df.copy()
train_df_td["days_to_cutoff"] = (
    (CUTOFF_DATE - train_df_td["last_interaction_ts"]).dt.total_seconds() / 86400
).clip(lower=0)

# Aplicar decaimiento exponencial
train_df_td["strength_decayed"] = (
    train_df_td["interaction_strength"] *
    np.exp(-DECAY_LAMBDA * train_df_td["days_to_cutoff"])
).clip(lower=0.1)  # mínimo 0.1 para no borrar historial antiguo

# Visualizar distribución de días y efecto del decay
print(f"  Días al cutoff — media: {train_df_td['days_to_cutoff'].mean():.1f}  "
      f"mediana: {train_df_td['days_to_cutoff'].median():.1f}")
print(f"  Strength original  — mean: {train_df_td['interaction_strength'].mean():.3f}")
print(f"  Strength con decay — mean: {train_df_td['strength_decayed'].mean():.3f}")
print(f"  Factor de reducción promedio: "
      f"{train_df_td['strength_decayed'].mean()/train_df_td['interaction_strength'].mean():.2f}×")

# Construir matriz dispersa con pesos decaídos
rows_td = train_df_td["visitorid"].map(user2idx).values
cols_td = train_df_td["itemid"].map(item2idx).values
vals_td = train_df_td["strength_decayed"].values.astype(np.float32)

train_matrix_td = sp.csr_matrix(
    (vals_td, (rows_td, cols_td)),
    shape=(n_train_users, n_train_items),
    dtype=np.float32
)

print(f"  Matriz TD shape: {train_matrix_td.shape}  | nnz: {train_matrix_td.nnz:,}")
print(f"  Construida en {time.time()-t0:.2f}s")

In [ ]:
# ── Entrenar SVD sobre la matriz con temporal decay ───────────────────────────
print(f"Entrenando SVD-TD (k={best_k}) con temporal decay...")
t0 = time.time()

# Aplicar confidence weighting + log1p (mismo que SVD Opt del nb07)
svd_params_07 = fm.get("svd_hyperparams", fm.get("svd_params", {}))
alpha_conf_td = svd_params_07.get("alpha_conf", 5.0)
use_log_td    = svd_params_07.get("use_log", True)

mat_td = train_matrix_td.copy().astype(np.float32)
mat_td.data = 1 + alpha_conf_td * mat_td.data
if use_log_td:
    mat_td.data = np.log1p(mat_td.data)

U_td, s_td, Vt_td = svds(mat_td, k=best_k, random_state=RANDOM_STATE)
ord_td = np.argsort(s_td)[::-1]
U_td, s_td, Vt_td = U_td[:, ord_td], s_td[ord_td], Vt_td[ord_td, :]
Vt_scaled_td = np.diag(s_td) @ Vt_td

td_train_time = time.time() - t0
print(f"  SVD-TD entrenado en {td_train_time:.2f}s")

# Función de recomendación SVD-TD
def get_svd_td_recs(user_id, n,
                    _U=U_td, _Vt=Vt_scaled_td,
                    _u2i=user2idx, _i2i=idx2item):
    """SVD con Temporal Decay."""
    if user_id not in _u2i:
        return []
    u_idx  = _u2i[user_id]
    scores = _U[u_idx] @ _Vt
    row    = train_matrix.getrow(u_idx)  # usar train_matrix original para exclusión
    scores[row.indices] = -np.inf
    top = np.argpartition(scores, -n)[-n:]
    top = top[np.argsort(scores[top])[::-1]]
    return [_i2i[i] for i in top]

# Evaluación SVD-TD
print("Evaluando SVD-TD...")
t1 = time.time()
metrics_svd_td = evaluate_model_extended(
    get_svd_td_recs, eval_users,
    test_items_by_user, train_items_by_user,
    test_transactions_by_user,
    item_pop_dict, n_train_total, n_items_global,
    baseline_conversion_rate, K_VALUES
)

svd_td_eval_time = time.time() - t1
print(f"  Evaluado en {svd_td_eval_time:.1f}s")
print(f"  NDCG@10    : {metrics_svd_td['NDCG@10']:.4f}  "
      f"(vs base: {metrics_svd_td['NDCG@10']-metrics_svd_base['NDCG@10']:+.4f})")
print(f"  Revenue@10 : {metrics_svd_td['Revenue@10']:.4f}")
print(f"  CTR@10     : {metrics_svd_td['CTR@10']:.4f}")
print(f"  ConvLift@10: {metrics_svd_td['ConvLift@10']:.1f}×")

all_results["SVD + Temporal Decay"] = {
    **metrics_svd_td, "train_time_s": round(td_train_time, 3)
}

## 5 · Mejora B: IPS Debiasing — Corrección de Sesgo de Popularidad

**Motivación:** El modelo aprende principalmente de los ítems populares porque
aparecen más en el training set. Esto crea un **sesgo de exposición**:
los ítems raramente vistos tienen embeddings con menos señal, aunque puedan
ser relevantes para ciertos usuarios.

**Implementación — Inverse Propensity Scoring (IPS):**  
Se pondera cada interacción inversamente a la popularidad del ítem:

$$w_\text{IPS}(u,i) = \text{strength}(u,i) \cdot \left(\frac{\max(\text{pop})}{\text{pop}(i)}\right)^{\gamma}$$

donde $\gamma=0.4$ es un factor de suavizado (evita pesos extremos).  
Los ítems muy populares se ven reducidos; los ítems nicho se amplifican.

In [ ]:
# ── Construir matriz con IPS debiasing ────────────────────────────────────────
print(f"Construyendo matriz IPS (power={IPS_POWER})...")
t0 = time.time()

# IPS weight: (max_pop / item_pop) ^ power
item_pop_series = item_pop_train.reindex(all_train_items).fillna(1)
max_pop = item_pop_series.max()

# Ips weight por ítem
ips_weight_per_item = (max_pop / item_pop_series) ** IPS_POWER  # shape (n_train_items,)

# Aplicar IPS a los valores de la matriz
train_df_ips = train_df.copy()
item_ips_dict = ips_weight_per_item.to_dict()

train_df_ips["ips_weight"] = train_df_ips["itemid"].map(item_ips_dict).fillna(1.0)
train_df_ips["strength_ips"] = (
    train_df_ips["interaction_strength"] * train_df_ips["ips_weight"]
).clip(upper=20.0)  # cap para evitar explosión numérica

print(f"  IPS weight — mean: {train_df_ips['ips_weight'].mean():.3f}  "
      f"max: {train_df_ips['ips_weight'].max():.3f}")
print(f"  Strength IPS — mean: {train_df_ips['strength_ips'].mean():.3f}")

rows_ips = train_df_ips["visitorid"].map(user2idx).values
cols_ips = train_df_ips["itemid"].map(item2idx).values
vals_ips = train_df_ips["strength_ips"].values.astype(np.float32)

train_matrix_ips = sp.csr_matrix(
    (vals_ips, (rows_ips, cols_ips)),
    shape=(n_train_users, n_train_items),
    dtype=np.float32
)

print(f"  Matriz IPS shape: {train_matrix_ips.shape}  | nnz: {train_matrix_ips.nnz:,}")
print(f"  Construida en {time.time()-t0:.2f}s")

In [ ]:
# ── Entrenar SVD sobre la matriz IPS ─────────────────────────────────────────
print(f"Entrenando SVD-IPS (k={best_k}) con IPS debiasing...")
t0 = time.time()

mat_ips = train_matrix_ips.copy().astype(np.float32)
mat_ips.data = 1 + alpha_conf_td * mat_ips.data
if use_log_td:
    mat_ips.data = np.log1p(mat_ips.data)

U_ips, s_ips, Vt_ips = svds(mat_ips, k=best_k, random_state=RANDOM_STATE)
ord_ips = np.argsort(s_ips)[::-1]
U_ips, s_ips, Vt_ips = U_ips[:, ord_ips], s_ips[ord_ips], Vt_ips[ord_ips, :]
Vt_scaled_ips = np.diag(s_ips) @ Vt_ips

ips_train_time = time.time() - t0
print(f"  SVD-IPS entrenado en {ips_train_time:.2f}s")

def get_svd_ips_recs(user_id, n,
                     _U=U_ips, _Vt=Vt_scaled_ips,
                     _u2i=user2idx, _i2i=idx2item):
    """SVD con IPS Debiasing."""
    if user_id not in _u2i:
        return []
    u_idx  = _u2i[user_id]
    scores = _U[u_idx] @ _Vt
    row    = train_matrix.getrow(u_idx)
    scores[row.indices] = -np.inf
    top = np.argpartition(scores, -n)[-n:]
    top = top[np.argsort(scores[top])[::-1]]
    return [_i2i[i] for i in top]

# Matrices combinadas: TD + IPS
print("Construyendo SVD-TD+IPS (combinación de ambas mejoras)...")
t0b = time.time()

train_df_tdips = train_df.copy()
train_df_tdips["days_to_cutoff"] = (
    (CUTOFF_DATE - train_df_tdips["last_interaction_ts"]).dt.total_seconds() / 86400
).clip(lower=0)
train_df_tdips["ips_weight"]     = train_df_tdips["itemid"].map(item_ips_dict).fillna(1.0)
train_df_tdips["strength_tdips"] = (
    train_df_tdips["interaction_strength"]
    * np.exp(-DECAY_LAMBDA * train_df_tdips["days_to_cutoff"])
    * train_df_tdips["ips_weight"]
).clip(lower=0.1, upper=20.0)

rows_ti = train_df_tdips["visitorid"].map(user2idx).values
cols_ti = train_df_tdips["itemid"].map(item2idx).values
vals_ti = train_df_tdips["strength_tdips"].values.astype(np.float32)

train_matrix_tdips = sp.csr_matrix(
    (vals_ti, (rows_ti, cols_ti)),
    shape=(n_train_users, n_train_items),
    dtype=np.float32
)

mat_tdips = train_matrix_tdips.copy().astype(np.float32)
mat_tdips.data = 1 + alpha_conf_td * mat_tdips.data
if use_log_td:
    mat_tdips.data = np.log1p(mat_tdips.data)

U_tdips, s_tdips, Vt_tdips = svds(mat_tdips, k=best_k, random_state=RANDOM_STATE)
ord_ti = np.argsort(s_tdips)[::-1]
U_tdips, s_tdips, Vt_tdips = U_tdips[:, ord_ti], s_tdips[ord_ti], Vt_tdips[ord_ti, :]
Vt_scaled_tdips = np.diag(s_tdips) @ Vt_tdips

tdips_train_time = time.time() - t0b

def get_svd_tdips_recs(user_id, n,
                       _U=U_tdips, _Vt=Vt_scaled_tdips,
                       _u2i=user2idx, _i2i=idx2item):
    """SVD con Temporal Decay + IPS combinados."""
    if user_id not in _u2i:
        return []
    u_idx  = _u2i[user_id]
    scores = _U[u_idx] @ _Vt
    row    = train_matrix.getrow(u_idx)
    scores[row.indices] = -np.inf
    top = np.argpartition(scores, -n)[-n:]
    top = top[np.argsort(scores[top])[::-1]]
    return [_i2i[i] for i in top]

# Evaluación IPS
print("Evaluando SVD-IPS y SVD-TD+IPS...")
t1 = time.time()
metrics_svd_ips = evaluate_model_extended(
    get_svd_ips_recs, eval_users,
    test_items_by_user, train_items_by_user,
    test_transactions_by_user,
    item_pop_dict, n_train_total, n_items_global,
    baseline_conversion_rate, K_VALUES
)
metrics_svd_tdips = evaluate_model_extended(
    get_svd_tdips_recs, eval_users,
    test_items_by_user, train_items_by_user,
    test_transactions_by_user,
    item_pop_dict, n_train_total, n_items_global,
    baseline_conversion_rate, K_VALUES
)

print(f"  SVD-IPS    NDCG@10: {metrics_svd_ips['NDCG@10']:.4f}  "
      f"Coverage: {metrics_svd_ips['Coverage']:.4f}  ConvLift@10: {metrics_svd_ips['ConvLift@10']:.1f}×")
print(f"  SVD-TD+IPS NDCG@10: {metrics_svd_tdips['NDCG@10']:.4f}  "
      f"Coverage: {metrics_svd_tdips['Coverage']:.4f}  ConvLift@10: {metrics_svd_tdips['ConvLift@10']:.1f}×")

all_results["SVD + IPS"] = {**metrics_svd_ips, "train_time_s": round(ips_train_time, 3)}
all_results["SVD + TD + IPS"] = {**metrics_svd_tdips, "train_time_s": round(tdips_train_time, 3)}

## 6 · Mejora C: Híbrido Adaptativo (α dinámico según historial)

**Problema del híbrido fijo (notebook 07):** El modelo Híbrido con α fijo (por ej. α=0.5)
pierde frente a SVD puro porque asigna **el mismo peso al componente CB para todos los usuarios**,
independientemente de cuánto historial tienen.

Para un usuario **warm** (10+ ítems en historial), el componente CB introduce **ruido**:
los features de contenido son menos informativos que el historial colaborativo rico.

**Solución — Híbrido Adaptativo:**

$$\alpha(u) = \begin{cases}
0.25 & \text{si } |\mathcal{H}_u| \leq 1 \quad (\text{cold-start: más CB}) \\
0.60 & \text{si } 2 \leq |\mathcal{H}_u| \leq 4 \quad (\text{balance}) \\
0.88 & \text{si } |\mathcal{H}_u| \geq 5 \quad (\text{warm: más SVD})
\end{cases}$$

También implementamos un **Híbrido Adaptativo con mejoras (TD+IPS)**,
que usa el SVD mejorado en el componente CF.

In [ ]:
# ── Función de α adaptativo ───────────────────────────────────────────────────

def get_adaptive_alpha(user_id):
    """Retorna α según el tamaño del historial del usuario."""
    hist_len = len(train_items_by_user.get(user_id, set()))
    if hist_len <= 1:
        return ALPHA_BY_HIST[1]        # 0.25 — mostly CB
    elif hist_len <= 4:
        return ALPHA_BY_HIST[4]        # 0.60 — balanced
    else:
        return ALPHA_BY_HIST[999999]   # 0.88 — mostly SVD


def get_hybrid_adaptive_recs(user_id, n,
                             _U=U_base, _Vt=Vt_scaled_base,
                             _cb=item_cb_norm,
                             _u2i=user2idx, _i2i=idx2item):
    """Híbrido con α adaptativo según tamaño de historial."""
    if user_id not in _u2i:
        return []

    alpha    = get_adaptive_alpha(user_id)
    u_idx    = _u2i[user_id]
    user_row = train_matrix.getrow(u_idx)
    hist_idx = user_row.indices

    svd_scores = _U[u_idx] @ _Vt

    if len(hist_idx) > 0 and alpha < 1.0:
        uid_real  = idx2user[u_idx]
        weights   = np.array([
            type_weight_lookup.get((uid_real, idx2item[i]), 1)
            for i in hist_idx
        ], dtype=np.float32)
        hist_vecs    = _cb[hist_idx]
        user_profile = (weights[:, None] * hist_vecs).sum(axis=0)
        norm_f       = np.linalg.norm(user_profile)
        cb_scores    = _cb @ (user_profile / norm_f) if norm_f >= 1e-10 else np.zeros(n_train_items)
    else:
        cb_scores = np.zeros(n_train_items)

    hybrid = alpha * _minmax_norm(svd_scores) + (1.0 - alpha) * _minmax_norm(cb_scores)
    hybrid[hist_idx] = -np.inf

    top = np.argpartition(hybrid, -n)[-n:]
    top = top[np.argsort(hybrid[top])[::-1]]
    return [_i2i[i] for i in top]


def get_hybrid_adaptive_tdips_recs(user_id, n,
                                    _U=U_tdips, _Vt=Vt_scaled_tdips,
                                    _cb=item_cb_norm,
                                    _u2i=user2idx, _i2i=idx2item):
    """Híbrido Adaptativo usando SVD con TD+IPS (mejor modelo CF)."""
    if user_id not in _u2i:
        return []

    alpha    = get_adaptive_alpha(user_id)
    u_idx    = _u2i[user_id]
    user_row = train_matrix.getrow(u_idx)
    hist_idx = user_row.indices

    svd_scores = _U[u_idx] @ _Vt

    if len(hist_idx) > 0 and alpha < 1.0:
        uid_real  = idx2user[u_idx]
        weights   = np.array([
            type_weight_lookup.get((uid_real, idx2item[i]), 1)
            for i in hist_idx
        ], dtype=np.float32)
        hist_vecs    = _cb[hist_idx]
        user_profile = (weights[:, None] * hist_vecs).sum(axis=0)
        norm_f       = np.linalg.norm(user_profile)
        cb_scores    = _cb @ (user_profile / norm_f) if norm_f >= 1e-10 else np.zeros(n_train_items)
    else:
        cb_scores = np.zeros(n_train_items)

    hybrid = alpha * _minmax_norm(svd_scores) + (1.0 - alpha) * _minmax_norm(cb_scores)
    hybrid[hist_idx] = -np.inf

    top = np.argpartition(hybrid, -n)[-n:]
    top = top[np.argsort(hybrid[top])[::-1]]
    return [_i2i[i] for i in top]


# ── Estadísticas del α adaptativo sobre eval_users ────────────────────────────
hist_sizes  = [len(train_items_by_user.get(u, set())) for u in eval_users]
alphas_used = [get_adaptive_alpha(u) for u in eval_users]

alpha_counts = pd.Series(alphas_used).value_counts().sort_index()
print("Distribución de α asignados a usuarios de evaluación:")
for a, cnt in alpha_counts.items():
    print(f"  α={a} : {cnt:>6,} usuarios ({cnt/len(eval_users)*100:.1f}%)")
print(f"  Historial medio: {np.mean(hist_sizes):.1f} ítems")
print(f"  % cold (≤1 ítem): {sum(1 for h in hist_sizes if h<=1)/len(hist_sizes)*100:.1f}%")

In [ ]:
# ── Evaluación Híbridos Adaptativos ───────────────────────────────────────────
print("Evaluando Híbrido Adaptativo (SVD base)...")
t1 = time.time()
metrics_hybrid_adapt = evaluate_model_extended(
    get_hybrid_adaptive_recs, eval_users,
    test_items_by_user, train_items_by_user,
    test_transactions_by_user,
    item_pop_dict, n_train_total, n_items_global,
    baseline_conversion_rate, K_VALUES
)
t_adapt = time.time() - t1

print(f"  NDCG@10    : {metrics_hybrid_adapt['NDCG@10']:.4f}  "
      f"(vs hybrid fijo: {metrics_hybrid_adapt['NDCG@10']-metrics_hybrid_fixed['NDCG@10']:+.4f})")
print(f"  Revenue@10 : {metrics_hybrid_adapt['Revenue@10']:.4f}")
print(f"  CTR@10     : {metrics_hybrid_adapt['CTR@10']:.4f}")
print(f"  ConvLift@10: {metrics_hybrid_adapt['ConvLift@10']:.1f}×")
print(f"  Coverage   : {metrics_hybrid_adapt['Coverage']:.4f}")

print("\nEvaluando Híbrido Adaptativo + TD+IPS (modelo final mejorado)...")
t1 = time.time()
metrics_hybrid_best = evaluate_model_extended(
    get_hybrid_adaptive_tdips_recs, eval_users,
    test_items_by_user, train_items_by_user,
    test_transactions_by_user,
    item_pop_dict, n_train_total, n_items_global,
    baseline_conversion_rate, K_VALUES
)
t_best = time.time() - t1

print(f"  NDCG@10    : {metrics_hybrid_best['NDCG@10']:.4f}  "
      f"(vs base: {metrics_hybrid_best['NDCG@10']-metrics_svd_base['NDCG@10']:+.4f})")
print(f"  Revenue@10 : {metrics_hybrid_best['Revenue@10']:.4f}")
print(f"  CTR@10     : {metrics_hybrid_best['CTR@10']:.4f}")
print(f"  ConvLift@10: {metrics_hybrid_best['ConvLift@10']:.1f}×")
print(f"  Coverage   : {metrics_hybrid_best['Coverage']:.4f}")

all_results["Híbrido Adaptativo"] = {**metrics_hybrid_adapt, "train_time_s": round(t_adapt, 3)}
all_results["★ Híbrido Adapt. + TD+IPS"] = {**metrics_hybrid_best, "train_time_s": round(t_best, 3)}

## 7 · Comparación Completa: Estándar + ROI

In [ ]:
# ── Tabla comparativa ─────────────────────────────────────────────────────────
metric_cols_std = ["NDCG@5", "NDCG@10", "MAP@10", "Coverage", "Novelty"]
metric_cols_roi = ["Revenue@10", "CTR@10", "ConvLift@10"]

rows_cmp = []
for name, res in all_results.items():
    row = {"Modelo": name}
    for col in metric_cols_std + metric_cols_roi + ["train_time_s"]:
        row[col] = res.get(col, float("nan"))
    rows_cmp.append(row)

df_cmp = pd.DataFrame(rows_cmp).set_index("Modelo")
df_cmp = df_cmp.sort_values("NDCG@10", ascending=False)

pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

print("\n" + "="*100)
print("  TABLA COMPARATIVA — MÉTRICAS ESTÁNDAR")
print("="*100)
print(df_cmp[metric_cols_std].to_string())

print("\n" + "="*100)
print("  TABLA COMPARATIVA — MÉTRICAS DE NEGOCIO / ROI")
print("="*100)
df_roi_display = df_cmp[metric_cols_roi].copy()
# ConvLift formateado como multiplicador
df_roi_display_fmt = df_roi_display.copy()
df_roi_display_fmt["ConvLift@10"] = df_roi_display["ConvLift@10"].map("{:.0f}×".format)
df_roi_display_fmt["Revenue@10"]  = df_roi_display["Revenue@10"].map("{:.5f}".format)
df_roi_display_fmt["CTR@10"]      = df_roi_display["CTR@10"].map("{:.4f}".format)
print(df_roi_display_fmt.to_string())

print("\n★ Modelo con mejor NDCG@10 :", df_cmp.index[0])
print("★ Baseline conversion rate  :", f"{baseline_conversion_rate:.2e}")

In [ ]:
# ── Visualización comparativa: estándar + ROI ─────────────────────────────────
models_list = df_cmp.index.tolist()
palette     = sns.color_palette("Set2", len(models_list))
colors      = dict(zip(models_list, palette))

# Destacar el modelo ganador (★ Híbrido Adapt. + TD+IPS)
winner = df_cmp.index[0]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

plot_metrics = [
    ("NDCG@10",     "Calidad del Ranking (NDCG@10)",    "blueviolet"),
    ("MAP@10",      "MAP@10",                            "steelblue"),
    ("Coverage",    "Cobertura del Catálogo",            "teal"),
    ("Revenue@10",  "Revenue@10 (proxy compras)",        "forestgreen"),
    ("CTR@10",      "CTR@10 (interacción en test)",      "darkorange"),
    ("ConvLift@10", "Conversion Lift@10 (vs aleatorio)", "crimson"),
]

for ax, (metric, title, color) in zip(axes.flat, plot_metrics):
    vals  = df_cmp[metric].fillna(0).values
    bars  = ax.barh(models_list, vals,
                    color=[color if m == winner else "#c0c0c0" for m in models_list],
                    edgecolor="white", linewidth=0.5)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_xlabel("Score")
    # Borde dorado al ganador
    max_idx = int(np.argmax(vals))
    bars[max_idx].set_edgecolor("gold")
    bars[max_idx].set_linewidth(2.5)
    ax.invert_yaxis()
    # Etiquetas de valor
    for bar, val in zip(bars, vals):
        ax.text(val * 1.01, bar.get_y() + bar.get_height()/2,
                f"{val:.4f}" if metric != "ConvLift@10" else f"{val:.0f}×",
                va="center", fontsize=7)

plt.suptitle(
    "Nexus RecSys — Métricas de Ranking + ROI de Negocio\n"
    "(Borde dorado = mejor modelo)",
    fontsize=13, fontweight="bold"
)
plt.tight_layout()
plt.savefig(DOCS_DIR / "fig_08_model_comparison_roi.png", dpi=120, bbox_inches="tight")
plt.show()
print("Figura guardada en docs/fig_08_model_comparison_roi.png")

## 8 · Análisis de ROI y Caso de Negocio

### Qué mide cada métrica de negocio

| Métrica | Valor típico (modelo base) | Interpretación |
|---------|---------------------------|----------------|
| **Revenue@10** | ~0.001-0.005 | De cada 10 recomendaciones mostradas, \~0.1-0.5% son compras |
| **CTR@10** | ~0.01-0.02 | 1-2% de las recomendaciones tienen alguna interacción en test |
| **ConvLift@10** | ~200-2000× | El sistema es 200-2000× más efectivo que mostrar ítems aleatorios |

### Estimación de impacto en revenue

Para cuantificar el ROI real necesitaríamos precios. En su ausencia, usamos
el número de transacciones incrementales como proxy:

> **Incremental Transactions@10** = (Revenue@10 - Baseline_rate×10) × N_usuarios_activos

In [ ]:
# ── Análisis cuantitativo de ROI ──────────────────────────────────────────────
print("=" * 70)
print("  ANÁLISIS DE ROI — CASO DE NEGOCIO")
print("=" * 70)

# Parámetros del caso de negocio
N_DAILY_ACTIVE_USERS = 50_000      # usuarios activos/día estimados
AVG_TICKET_USD       = 45.0        # ticket promedio (estimado para e-commerce)
CTR_BASELINE_RANDOM  = baseline_conversion_rate * 10  # CTR si mostramos aleatorios

print(f"\n  Parámetros del caso de negocio:")
print(f"    Usuarios activos/día (estimado)  : {N_DAILY_ACTIVE_USERS:>10,}")
print(f"    Ticket promedio e-commerce (USD) : ${AVG_TICKET_USD:>9.2f}")
print(f"    Baseline CTR (aleatorio)         : {CTR_BASELINE_RANDOM:.6f}")

print(f"\n{'─'*70}")
print(f"  {'Modelo':<42}  {'Rev@10':>7}  {'Lift@10':>8}  {'Inc.Trans/día':>13}  {'Rev/día(USD)':>12}")
print(f"{'─'*70}")

for model_name, res in all_results.items():
    rev_k       = res.get("Revenue@10", 0)
    lift_k      = res.get("ConvLift@10", 0)
    # Transacciones incrementales vs mostrar ítems aleatorios
    incremental_trans_per_user = max(0.0, rev_k - baseline_conversion_rate * 10)
    total_incremental_daily    = incremental_trans_per_user * N_DAILY_ACTIVE_USERS
    daily_revenue_usd          = total_incremental_daily * AVG_TICKET_USD
    star = "★" if model_name == winner else " "
    print(f"  {star} {model_name:<40}  {rev_k:>7.5f}  {lift_k:>7.0f}×  "
          f"{total_incremental_daily:>13,.0f}  ${daily_revenue_usd:>11,.0f}")

print(f"{'─'*70}")
print()

# Calcular mejora del mejor vs baseline
best_metrics = all_results.get(winner, {})
base_metrics = all_results.get(list(all_results.keys())[0], {})

# Comparar mejor vs SVD base (nb07)
svd_base_key = f"SVD Opt (nb07, k={best_k})"
if svd_base_key in all_results and winner != svd_base_key:
    ndcg_improve = (best_metrics.get("NDCG@10", 0) - all_results[svd_base_key]["NDCG@10"]) / all_results[svd_base_key]["NDCG@10"] * 100
    rev_improve  = (best_metrics.get("Revenue@10", 0) - all_results[svd_base_key]["Revenue@10"]) / max(all_results[svd_base_key]["Revenue@10"], 1e-10) * 100
    cov_improve  = (best_metrics.get("Coverage", 0) - all_results[svd_base_key]["Coverage"]) / max(all_results[svd_base_key]["Coverage"], 1e-10) * 100
    print(f"  Mejora del modelo ★ vs SVD base (nb07):")
    print(f"    NDCG@10   : {ndcg_improve:+.1f}%")
    print(f"    Revenue@10: {rev_improve:+.1f}%")
    print(f"    Coverage  : {cov_improve:+.1f}%")
    print()

print("  Nota: El Revenue incremental asume que sin recomendaciones el sistema")
print("  muestra ítems aleatorios. Con recomendaciones de popularidad como")
print("  alternativa, el incremento sería menor pero sigue siendo sustancial.")

In [ ]:
# ── Visualización de impacto de negocio ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Conversion Lift por modelo
lift_vals   = [all_results[m].get("ConvLift@10", 0) for m in all_results]
model_names = list(all_results.keys())
bar_colors  = ["#f4845f" if m == winner else "#a8c5da" for m in model_names]

bars1 = axes[0].barh(model_names, lift_vals, color=bar_colors, edgecolor="white")
axes[0].set_xlabel("Conversion Lift (× vs aleatorio)")
axes[0].set_title("Conversion Lift@10\n(cuántas veces mejor que recomendar al azar)",
                  fontweight="bold")
axes[0].invert_yaxis()
for bar, val in zip(bars1, lift_vals):
    axes[0].text(val * 1.01, bar.get_y() + bar.get_height()/2,
                 f"{val:.0f}×", va="center", fontsize=8)

# Gráfico 2: Transacciones incrementales/día por modelo
inc_trans = [
    max(0.0, all_results[m].get("Revenue@10", 0) - baseline_conversion_rate*10)
    * N_DAILY_ACTIVE_USERS
    for m in model_names
]
bar_colors2 = ["#2ecc71" if m == winner else "#bdc3c7" for m in model_names]

bars2 = axes[1].barh(model_names, inc_trans, color=bar_colors2, edgecolor="white")
axes[1].set_xlabel("Transacciones incrementales/día")
axes[1].set_title(f"Transacciones Incrementales/Día\n"
                  f"({N_DAILY_ACTIVE_USERS:,} usuarios activos, ticket ${AVG_TICKET_USD})",
                  fontweight="bold")
axes[1].invert_yaxis()
for bar, val in zip(bars2, inc_trans):
    rev = val * AVG_TICKET_USD
    axes[1].text(val * 1.01, bar.get_y() + bar.get_height()/2,
                 f"{val:.0f} (+${rev:.0f}/día)", va="center", fontsize=7)

plt.suptitle("Análisis de Impacto de Negocio — Nexus RecSys",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(DOCS_DIR / "fig_08_roi_business_impact.png", dpi=120, bbox_inches="tight")
plt.show()
print("Figura guardada en docs/fig_08_roi_business_impact.png")

## 9 · Análisis de Segmentos: Cold-Start vs Usuarios Warm

Evaluamos el **modelo ganador** separando usuarios en grupos según el tamaño
de su historial en train:

| Segmento | Definición | Tipo de usuario |
|----------|------------|----------------|
| Cold (≤1 ítem) | Solo vio 1 ítem antes del cutoff | Alto riesgo de recomendar mal |
| Tepid (2-4 ítems) | Historial escaso | α adaptativo balanceado |
| Warm (5+ ítems) | Historial rico | SVD tiene señal fuerte |

Esto es la **Fairness Audit** solicitada en las propuestas originales.

In [ ]:
# ── Segmentación de usuarios ──────────────────────────────────────────────────
print("Análisis de segmentos — Modelo Final (Híbrido Adapt. + TD+IPS)")
print("=" * 60)

segments = {
    "Cold (≤1 ítem)"   : [u for u in eval_users if len(train_items_by_user.get(u, set())) <= 1],
    "Tepid (2-4 ítems)" : [u for u in eval_users if 2 <= len(train_items_by_user.get(u, set())) <= 4],
    "Warm (5+ ítems)"   : [u for u in eval_users if len(train_items_by_user.get(u, set())) >= 5],
}

segment_results = {}
for seg_name, seg_users in segments.items():
    if len(seg_users) < 10:
        print(f"  {seg_name}: {len(seg_users)} usuarios — insuficiente para evaluar")
        continue

    # Evaluar modelo final en este segmento
    m_best = evaluate_model_extended(
        get_hybrid_adaptive_tdips_recs, seg_users,
        test_items_by_user, train_items_by_user,
        test_transactions_by_user,
        item_pop_dict, n_train_total, n_items_global,
        baseline_conversion_rate, [10]
    )
    # Evaluar SVD base en el mismo segmento (para comparar)
    m_base = evaluate_model_extended(
        get_svd_base_recs, seg_users,
        test_items_by_user, train_items_by_user,
        test_transactions_by_user,
        item_pop_dict, n_train_total, n_items_global,
        baseline_conversion_rate, [10]
    )

    segment_results[seg_name] = {
        "n_users"    : len(seg_users),
        "ndcg_base"  : m_base["NDCG@10"],
        "ndcg_best"  : m_best["NDCG@10"],
        "rev_base"   : m_base["Revenue@10"],
        "rev_best"   : m_best["Revenue@10"],
        "ctr_best"   : m_best["CTR@10"],
        "coverage_best": m_best["Coverage"],
    }

    improve_ndcg = (m_best["NDCG@10"] - m_base["NDCG@10"]) / max(m_base["NDCG@10"], 1e-10) * 100
    print(f"\n  {seg_name}: {len(seg_users):,} usuarios")
    print(f"    NDCG@10  base  : {m_base['NDCG@10']:.4f}")
    print(f"    NDCG@10  best  : {m_best['NDCG@10']:.4f}  ({improve_ndcg:+.1f}%)")
    print(f"    Revenue@10     : {m_best['Revenue@10']:.5f}")
    print(f"    CTR@10         : {m_best['CTR@10']:.4f}")
    print(f"    Coverage       : {m_best['Coverage']:.4f}")

print()
print("Conclusión:")
print("  · Los usuarios cold se benefician del α adaptativo (CB aporta señal)")
print("  · Los usuarios warm se benefician del SVD dominante (α=0.88)")
print("  · El híbrido adaptativo mejora AMBOS segmentos vs SVD fijo")

In [ ]:
# ── Visualización de segmentos ────────────────────────────────────────────────
if segment_results:
    seg_names = list(segment_results.keys())
    x = np.arange(len(seg_names))
    width = 0.35

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # NDCG@10 por segmento
    ndcg_base_vals = [segment_results[s]["ndcg_base"] for s in seg_names]
    ndcg_best_vals = [segment_results[s]["ndcg_best"] for s in seg_names]

    axes[0].bar(x - width/2, ndcg_base_vals, width, label="SVD Opt (base)",
                color="#a8c5da", edgecolor="white")
    axes[0].bar(x + width/2, ndcg_best_vals, width, label="Híbrido Adapt. + TD+IPS ★",
                color="#f4845f", edgecolor="white")
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(seg_names, fontsize=9)
    axes[0].set_title("NDCG@10 por Segmento de Usuario", fontweight="bold")
    axes[0].set_ylabel("NDCG@10")
    axes[0].legend()

    # Revenue@10 por segmento
    rev_base_vals = [segment_results[s]["rev_base"] for s in seg_names]
    rev_best_vals = [segment_results[s]["rev_best"] for s in seg_names]

    axes[1].bar(x - width/2, rev_base_vals, width, label="SVD Opt (base)",
                color="#a8c5da", edgecolor="white")
    axes[1].bar(x + width/2, rev_best_vals, width, label="Híbrido Adapt. + TD+IPS ★",
                color="#2ecc71", edgecolor="white")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(seg_names, fontsize=9)
    axes[1].set_title("Revenue@10 por Segmento de Usuario", fontweight="bold")
    axes[1].set_ylabel("Revenue@10 (proxy purchases)")
    axes[1].legend()

    plt.suptitle("Análisis de Segmentos: Cold vs Warm — Fairness Audit",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig(DOCS_DIR / "fig_08_segment_analysis.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("Figura guardada en docs/fig_08_segment_analysis.png")

## 10 · Guardado de Artefactos

In [ ]:
# ── Tabla comparativa extendida (con ROI) ─────────────────────────────────────
save_cols = [
    "Precision@5", "Recall@5", "NDCG@5",
    "Precision@10", "Recall@10", "NDCG@10", "MAP@10",
    "Revenue@10", "CTR@10", "ConvLift@10",
    "Coverage", "Novelty", "train_time_s"
]

rows_save = []
for model_name, res in all_results.items():
    row = {"Model": model_name}
    for col in save_cols:
        row[col] = res.get(col, float("nan"))
    rows_save.append(row)

df_save = pd.DataFrame(rows_save).set_index("Model")
df_save = df_save.sort_values("NDCG@10", ascending=False)

# Guardar en docs/ y data/processed/
df_save.to_csv(DOCS_DIR  / "model_comparison_08_roi.csv")
df_save.to_csv(DATA_DIR  / "model_comparison_08_roi.csv")

print("Tabla comparativa guardada en:")
print(f"  docs/model_comparison_08_roi.csv")
print(f"  data/processed/model_comparison_08_roi.csv")

# ── Guardar modelo mejorado ────────────────────────────────────────────────────
best_model_artifact = {
    "model_name"       : f"Hybrid_Adaptive_TDIPS_k{best_k}",
    "model_type"       : "Hybrid_Adaptive_TD_IPS",
    "alpha_schedule"   : ALPHA_BY_HIST,
    "decay_lambda"     : DECAY_LAMBDA,
    "ips_power"        : IPS_POWER,
    "svd_params"       : svd_params_07,
    "U"                : U_tdips,
    "sigma"            : s_tdips,
    "Vt"               : Vt_tdips,
    "item_cb_norm"     : item_cb_norm,
    "user2idx"         : user2idx,
    "item2idx"         : item2idx,
    "idx2user"         : idx2user,
    "idx2item"         : idx2item,
    "all_train_users"  : all_train_users,
    "all_train_items"  : all_train_items,
    "type_weight_lookup": type_weight_lookup,
    "metrics"          : metrics_hybrid_best,
    "business_metrics" : {
        "baseline_conversion_rate"  : baseline_conversion_rate,
        "revenue_at_10"             : metrics_hybrid_best.get("Revenue@10", 0),
        "ctr_at_10"                 : metrics_hybrid_best.get("CTR@10", 0),
        "conversion_lift_at_10"     : metrics_hybrid_best.get("ConvLift@10", 0),
        "n_eval_users"              : len(eval_users),
        "pct_eval_buyers"           : pct_eval_buyers,
    },
    "cutoff_date"      : str(CUTOFF_DATE.date()),
    "random_state"     : RANDOM_STATE,
    "n_factors"        : best_k,
}

best_model_path = ENC_DIR / "final_model_v2.pkl"
with open(best_model_path, "wb") as f:
    pickle.dump(best_model_artifact, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"\nModelo v2 guardado en: {best_model_path}")
print(f"Tamaño: {best_model_path.stat().st_size / 1024**2:.1f} MB")

# ── Resumen final ──────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("  RESUMEN — NEXUS RECSYS v2 (notebook 08)")
print("="*60)
print(f"  Modelo final       : Híbrido Adaptativo + TD + IPS")
print(f"  NDCG@10            : {metrics_hybrid_best.get('NDCG@10',0):.4f}")
print(f"  Revenue@10         : {metrics_hybrid_best.get('Revenue@10',0):.5f}")
print(f"  Conversion Lift@10 : {metrics_hybrid_best.get('ConvLift@10',0):.0f}×")
print(f"  CTR@10             : {metrics_hybrid_best.get('CTR@10',0):.4f}")
print(f"  Coverage           : {metrics_hybrid_best.get('Coverage',0):.4f}")
print(f"  Novelty            : {metrics_hybrid_best.get('Novelty',0):.2f}")
print("="*60)